# 🧠 Project Kaizen — Agentic Kernel Training
### Meta × Scaler OpenEnv Hackathon

**Theme 3.1 — World Modeling (Professional Tasks)** | **Theme 2 — Long-Horizon Planning**

This notebook trains a Qwen-2.5 3B agent to autonomously manage a simulated OS:
1. **SFT** — Supervised Fine-Tuning on 60 hand-crafted golden examples (Unsloth + LoRA)
2. **GRPO** — Group Relative Policy Optimisation against a live KaizenEnv reward signal (TRL)

**Continual improvement:** Run cells again after the first training — the scripts auto-detect existing adapters and keep improving them. Every run saves checkpoints so a Colab crash never loses progress.

---
| Setting | Value |
|---|---|
| Runtime | **T4 GPU (free tier, 16 GB VRAM)** |
| Model | Qwen2.5-3B-Instruct (4-bit, ~2.5 GB) |
| SFT steps | 100 |
| GRPO steps | 80 |
| Group size | 4 completions / prompt |


## 0️⃣  Verify GPU Runtime
> ⚠️ **You must be on a T4 GPU.** Runtime → Change runtime type → T4 GPU

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode != 0:
    print("❌  No GPU detected. Go to Runtime → Change runtime type → T4 GPU and restart.")
    sys.exit(1)

gpu_info = result.stdout.strip()
print(f"✅  GPU detected: {gpu_info}")

import torch
print(f"✅  PyTorch CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"    VRAM total : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"    VRAM free  : {torch.cuda.memory_reserved(0) / 1e9:.2f} GB reserved so far")


## 1️⃣  Install Dependencies

This takes ~3-5 minutes on a fresh Colab runtime. Skip if already installed.

In [ ]:
%%capture install_output
# Core ML stack
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
# Environment + server deps
!pip install gymnasium psutil pydantic>=2.0.0 fastapi uvicorn websockets
# Plotting
!pip install matplotlib

print("✅  Installation complete")


## 2️⃣  Set Up Project Files

**Option A — Clone from GitHub (recommended):**  Fill in your repo URL below.

**Option B — Upload a zip:**  Upload `kaizen_os.zip` to Colab's file browser, then set `USE_ZIP = True`.


In [ ]:
import os, zipfile

# ── Configuration ────────────────────────────────────────────────────────────
GITHUB_REPO = ""          # e.g. "https://github.com/yourname/kaizen_os"
USE_ZIP     = False       # set True if you uploaded kaizen_os.zip instead
ZIP_PATH    = "/content/kaizen_os.zip"
PROJECT_DIR = "/content/kaizen_os"
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(PROJECT_DIR):
    print(f"📁  Project already at {PROJECT_DIR} — skipping clone/unzip.")

elif GITHUB_REPO:
    print(f"🔗  Cloning {GITHUB_REPO} ...")
    os.system(f"git clone {GITHUB_REPO} {PROJECT_DIR}")
    print("✅  Clone complete.")

elif USE_ZIP:
    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(
            f"ZIP not found at {ZIP_PATH}. "
            "Upload kaizen_os.zip using the Files panel on the left."
        )
    print(f"📦  Unzipping {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/content/")
    # Handle zips that add an extra top-level folder
    if not os.path.isdir(PROJECT_DIR):
        dirs = [d for d in os.listdir("/content/") if os.path.isdir(f"/content/{d}") and "kaizen" in d.lower()]
        if dirs:
            os.rename(f"/content/{dirs[0]}", PROJECT_DIR)
    print("✅  Unzip complete.")

else:
    raise ValueError(
        "No source configured. Set GITHUB_REPO to your repo URL, "
        "or upload kaizen_os.zip and set USE_ZIP = True."
    )

# Verify critical files exist
required = [
    "training/golden_examples.json",
    "training/sft_train.py",
    "training/grpo_train.py",
    "environment/kaizen_env.py",
    "environment/action_space.py",
]
missing = [f for f in required if not os.path.isfile(os.path.join(PROJECT_DIR, f))]
if missing:
    print(f"⚠️  Missing files: {missing}")
    print("   Make sure your project zip/repo contains the full kaizen_os/ structure.")
else:
    print(f"✅  All required files present in {PROJECT_DIR}")


## 3️⃣  Configure Python Path

In [ ]:
import sys, os

PROJECT_DIR = "/content/kaizen_os"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Quick smoke test — import environment modules
try:
    from environment.action_space import parse_action, WaitAction
    from environment.chaos import ChaosInjector
    from environment.reward import compute_reward
    print("✅  Environment modules import OK")
except ImportError as e:
    print(f"❌  Import error: {e}")
    print("    Check that PROJECT_DIR points to the root of kaizen_os/")


## 4️⃣  Inspect Golden Examples (SFT Dataset)

In [ ]:
import json, os

DATASET_PATH = os.path.join(PROJECT_DIR, "training/golden_examples.json")

with open(DATASET_PATH) as f:
    examples = json.load(f)

print(f"📊  Total examples : {len(examples)}")

# Count by chaos type (detected from instruction text)
chaos_counts = {"memory_leak": 0, "cpu_hog": 0, "thermal_spike": 0, "healthy": 0}
for ex in examples:
    ins = ex["instruction"]
    if "memory_leak" in ins:
        chaos_counts["memory_leak"] += 1
    elif "cpu_hog" in ins or "rogue_compute" in ins:
        chaos_counts["cpu_hog"] += 1
    elif "thermal" in ins.lower():
        chaos_counts["thermal_spike"] += 1
    else:
        chaos_counts["healthy"] += 1

print("\nDistribution:")
for k, v in chaos_counts.items():
    bar = "█" * v
    print(f"  {k:<15} {v:>3}  {bar}")

print("\n─── Example #1 (instruction preview) ───")
print(examples[0]["instruction"][:500])
print("\n─── Example #1 (output preview) ───")
print(examples[0]["output"][:300])


## 5️⃣  (Optional) Mount Google Drive for Persistent Storage

Skip this cell if you don't need to save models across sessions.  
If mounted, models will be saved to Drive so you never lose them when Colab resets.


In [ ]:
USE_DRIVE = False   # ← set True to enable Drive mounting

DRIVE_SFT_PATH  = "/content/drive/MyDrive/kaizen_os/kaizen_sft_model"
DRIVE_GRPO_PATH = "/content/drive/MyDrive/kaizen_os/kaizen_grpo_model"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    os.makedirs(DRIVE_SFT_PATH,  exist_ok=True)
    os.makedirs(DRIVE_GRPO_PATH, exist_ok=True)

    # Symlink so training scripts write directly to Drive
    SFT_OUTPUT  = DRIVE_SFT_PATH
    GRPO_OUTPUT = DRIVE_GRPO_PATH

    print(f"✅  Drive mounted. Models will save to:")
    print(f"    SFT  → {SFT_OUTPUT}")
    print(f"    GRPO → {GRPO_OUTPUT}")

else:
    SFT_OUTPUT  = "/content/kaizen_os/kaizen_sft_model"
    GRPO_OUTPUT = "/content/kaizen_os/kaizen_grpo_model"
    print("ℹ️  Drive not mounted. Models save to Colab local storage.")
    print("    ⚠️  These will be lost when the runtime resets!")
    print(f"    SFT  → {SFT_OUTPUT}")
    print(f"    GRPO → {GRPO_OUTPUT}")


## 6️⃣  SFT Training — Supervised Fine-Tuning

**What this does:**
- Loads Qwen2.5-3B-Instruct in 4-bit (saves ~6 GB VRAM)  
- Attaches a LoRA adapter (rank 16) — only 1.2% of weights are trainable  
- Trains 100 steps on the 60 golden examples  
- Saves checkpoints every 25 steps (crash-safe)

**Continual improvement:** If `kaizen_sft_model/` already exists, this cell loads that adapter and trains on top of it instead of starting from scratch.

⏱️ Expected time on T4: ~8–12 minutes


In [ ]:
import os, sys

PROJECT_DIR = "/content/kaizen_os"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# ── Optionally override output directory if Drive is mounted ─────────────────
try:
    SFT_OUTPUT
except NameError:
    SFT_OUTPUT = os.path.join(PROJECT_DIR, "kaizen_sft_model")

os.environ["KAIZEN_SFT_OUTPUT"] = SFT_OUTPUT

# ── Patch sft_train.py OUTPUT_DIR at import time ─────────────────────────────
import importlib, types

sft_spec = importlib.util.spec_from_file_location(
    "sft_train",
    os.path.join(PROJECT_DIR, "training/sft_train.py")
)
sft_module = importlib.util.module_from_spec(sft_spec)

# Override OUTPUT_DIR before the module executes
sft_module.__dict__["OUTPUT_DIR"]      = SFT_OUTPUT
sft_module.__dict__["CHECKPOINT_DIR"]  = os.path.join(SFT_OUTPUT, "checkpoints")
sft_module.__dict__["RUN_HISTORY_PATH"]= os.path.join(SFT_OUTPUT, "run_history.json")
sft_module.__dict__["DATASET_PATH"]    = os.path.join(PROJECT_DIR, "training/golden_examples.json")

sft_spec.loader.exec_module(sft_module)

# Run training
output_path = sft_module.train()
print(f"\n🎉  SFT complete. Adapter at: {output_path}")


### 📈 SFT Loss Curves

In [ ]:
import os, glob
from IPython.display import display, Image as IPImage

try:
    SFT_OUTPUT
except NameError:
    SFT_OUTPUT = "/content/kaizen_os/kaizen_sft_model"

# Show all individual run plots
run_plots = sorted(glob.glob(os.path.join(SFT_OUTPUT, "sft_loss_curve_run*.png")))
combined  = os.path.join(SFT_OUTPUT, "sft_combined_runs.png")

if not run_plots and not os.path.isfile(combined):
    print("No plots found yet — run the SFT training cell first.")
else:
    for p in run_plots:
        print(f"Run plot: {p}")
        display(IPImage(p, width=800))

    if os.path.isfile(combined) and len(run_plots) > 1:
        print(f"\nCombined: {combined}")
        display(IPImage(combined, width=800))


### 🔬 Quick SFT Inference Smoke Test

Run a single inference pass to verify the adapter produces valid JSON actions before committing to GRPO.


In [ ]:
import os, sys, json, torch

PROJECT_DIR = "/content/kaizen_os"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

try:
    SFT_OUTPUT
except NameError:
    SFT_OUTPUT = os.path.join(PROJECT_DIR, "kaizen_sft_model")

from agent.llm_agent import LLMAgent
from agent.prompts import format_observation
from environment.action_space import parse_action
from environment.chaos import CHAOS_SCENARIOS

# Build a mock chaos observation
mock_obs = {
    "cpu_percent": 91.0,
    "ram_percent": 74.0,
    "thermal_celsius": 88.0,
    "uptime_seconds": 43.0,
    "active_chaos": "memory_leak",
    "step": 3,
    "log_snippet": CHAOS_SCENARIOS["memory_leak"]["log"],
    "process_list": [
        {"pid": 1,    "name": "kernel_task",    "cpu_percent": 2.1,  "memory_mb": 80.0,  "status": "running", "is_protected": True},
        {"pid": 88,   "name": "nav_service",    "cpu_percent": 1.4,  "memory_mb": 40.0,  "status": "running", "is_protected": True},
        {"pid": 2847, "name": "memory_leak_sim","cpu_percent": 67.4, "memory_mb": 512.0, "status": "running", "is_protected": False},
        {"pid": 312,  "name": "python3",        "cpu_percent": 4.2,  "memory_mb": 220.0, "status": "running", "is_protected": False},
    ],
}

print("Loading SFT adapter for smoke test...")
agent = LLMAgent(model_name=SFT_OUTPUT)

print("\nRunning inference...")
raw_output, action, error = agent.act(mock_obs)

print("\n─── Raw LLM output ───")
print(raw_output[:800])

print("\n─── Parsed action ───")
if action:
    print(f"✅  Tool    : {action.tool_name}")
    print(f"    Action  : {action.model_dump()}")
    if hasattr(action, "pid"):
        pid_ok = action.pid not in {1, 88}
        print(f"    PID safe: {'✅' if pid_ok else '❌ PROTECTED PID!'}")
else:
    print(f"❌  Parse failed: {error}")
    print("    This will be penalised -1.0 reward during GRPO.")


## 7️⃣  GRPO Training — Reinforcement Learning

**What this does:**
- Loads the SFT adapter (or existing GRPO policy on subsequent runs)
- Generates 4 completions per prompt, scores each against the live `KaizenEnv`
- Uses within-group relative reward as the advantage — no separate critic needed
- Saves checkpoints every 20 steps

**Reward signal:**
| Event | Reward |
|---|---|
| Parse failure (invalid JSON) | -1.0 |
| Kill protected process | -10.0 |
| Chaos resolved | +3.0 |
| CPU/thermal improvement | +0.1 – 0.5 |
| System nominal (cpu<40, thermal<70) | +1.0 |
| Kill idle process (cpu<5%) | -2.0 |

⏱️ Expected time on T4: ~15–20 minutes


In [ ]:
import os, sys

PROJECT_DIR = "/content/kaizen_os"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

try:
    SFT_OUTPUT
    GRPO_OUTPUT
except NameError:
    SFT_OUTPUT  = os.path.join(PROJECT_DIR, "kaizen_sft_model")
    GRPO_OUTPUT = os.path.join(PROJECT_DIR, "kaizen_grpo_model")

# ── Load and patch grpo_train module ─────────────────────────────────────────
import importlib.util

grpo_spec = importlib.util.spec_from_file_location(
    "grpo_train",
    os.path.join(PROJECT_DIR, "training/grpo_train.py")
)
grpo_module = importlib.util.module_from_spec(grpo_spec)

grpo_module.__dict__["MODEL_PATH"]       = SFT_OUTPUT
grpo_module.__dict__["OUTPUT_DIR"]       = GRPO_OUTPUT
grpo_module.__dict__["CHECKPOINT_DIR"]   = os.path.join(GRPO_OUTPUT, "checkpoints")
grpo_module.__dict__["RUN_HISTORY_PATH"] = os.path.join(GRPO_OUTPUT, "run_history.json")
grpo_module.__dict__["DATASET_PATH"]     = os.path.join(PROJECT_DIR, "training/golden_examples.json")

grpo_spec.loader.exec_module(grpo_module)

# Run GRPO
output_path = grpo_module.train()
print(f"\n🎉  GRPO complete. Policy at: {output_path}")


### 📈 GRPO Reward Curves

In [ ]:
import os, glob
from IPython.display import display, Image as IPImage

try:
    GRPO_OUTPUT
except NameError:
    GRPO_OUTPUT = "/content/kaizen_os/kaizen_grpo_model"

run_plots = sorted(glob.glob(os.path.join(GRPO_OUTPUT, "grpo_reward_curve_run*.png")))
combined  = os.path.join(GRPO_OUTPUT, "grpo_combined_runs.png")

if not run_plots and not os.path.isfile(combined):
    print("No plots found yet — run the GRPO training cell first.")
else:
    for p in run_plots:
        print(f"Run plot: {p}")
        display(IPImage(p, width=800))

    if os.path.isfile(combined) and len(run_plots) > 1:
        print(f"\nCombined: {combined}")
        display(IPImage(combined, width=800))


### 📋 Full Run History

In [ ]:
import json, os

try:
    SFT_OUTPUT
    GRPO_OUTPUT
except NameError:
    SFT_OUTPUT  = "/content/kaizen_os/kaizen_sft_model"
    GRPO_OUTPUT = "/content/kaizen_os/kaizen_grpo_model"

def _print_history(label, path):
    if not os.path.isfile(path):
        print(f"{label}: no history yet.")
        return
    with open(path) as f:
        history = json.load(f)
    print(f"\n{'─'*50}")
    print(f"{label} — {len(history)} run(s)")
    print(f"{'─'*50}")
    for r in history:
        run_num  = r["run"]
        ts       = r.get("timestamp", "unknown")
        runtime  = r.get("runtime_s", 0)
        mode     = "⟳ continual" if r.get("continual") else "✦ fresh"
        resumed  = f" (resumed from ckpt)" if r.get("resumed_from") else ""

        if "final_loss" in r:
            metric = f"final loss {r['final_loss']:.4f}"
        else:
            metric = f"final reward {r.get('final_reward', 0):+.4f}"

        print(f"  Run {run_num:02d} | {ts} | {mode}{resumed}")
        print(f"         {metric} | runtime {runtime:.0f}s")

_print_history("SFT",  os.path.join(SFT_OUTPUT,  "run_history.json"))
_print_history("GRPO", os.path.join(GRPO_OUTPUT, "run_history.json"))


## 8️⃣  Evaluate — Full Episode Rollout

Run a complete 10-step episode to see the trained agent in action. Outputs the chain-of-thought reasoning, chosen action, and reward at each step.


In [ ]:
import os, sys, json

PROJECT_DIR = "/content/kaizen_os"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

try:
    GRPO_OUTPUT
except NameError:
    GRPO_OUTPUT = os.path.join(PROJECT_DIR, "kaizen_grpo_model")

# Fall back to SFT model if GRPO hasn't run yet
MODEL_TO_EVAL = GRPO_OUTPUT if os.path.isfile(os.path.join(GRPO_OUTPUT, "adapter_config.json")) else SFT_OUTPUT

print(f"Evaluating model: {MODEL_TO_EVAL}")
print("─" * 60)

from agent.llm_agent import LLMAgent
from environment.kaizen_env import KaizenEnv

agent = LLMAgent(model_name=MODEL_TO_EVAL)
env   = KaizenEnv(broadcast=False)

obs, info = env.reset()
cumulative_reward = 0.0
step = 0

print(f"Episode started. Max steps: {env.MAX_STEPS}\n")

while True:
    step += 1
    print(f"{'='*60}")
    print(f"STEP {step:02d}/{env.MAX_STEPS}")
    print(f"  CPU: {obs['cpu_percent']:.1f}%  RAM: {obs['ram_percent']:.1f}%  "
          f"Thermal: {obs['thermal_celsius']:.1f}°C")
    print(f"  Active chaos: {obs['active_chaos'] or 'None'}")

    raw_output, action, error = agent.act(obs)

    # Print reasoning (first 400 chars)
    reasoning_end = raw_output.rfind("{")
    reasoning = raw_output[:reasoning_end].strip() if reasoning_end > 0 else ""
    if reasoning:
        print(f"\n  Reasoning: {reasoning[:400]}{'...' if len(reasoning)>400 else ''}")

    if action:
        print(f"\n  Action: {action.model_dump()}")
    else:
        print(f"\n  ⚠️  Parse error: {error} → reward -1.0")

    obs, reward, terminated, truncated, info = env.step(raw_output)
    cumulative_reward += reward

    print(f"\n  Reward: {reward:+.3f}  |  Cumulative: {cumulative_reward:+.3f}")

    if terminated:
        print(f"\n{'='*60}")
        if obs["active_chaos"] is None:
            print(f"✅  CHAOS RESOLVED in {step} steps! Cumulative reward: {cumulative_reward:+.3f}")
        else:
            print(f"🏁  Episode ended (max steps). Cumulative reward: {cumulative_reward:+.3f}")
        break
    if truncated:
        print(f"\n{'='*60}")
        print(f"⏱️  Episode truncated at max steps. Cumulative reward: {cumulative_reward:+.3f}")
        break


## 9️⃣  Benchmark — 10 Episodes

Run 10 episodes and report resolution rate, average reward, and average steps to resolution. Use this to compare before/after GRPO.


In [ ]:
import os, sys

PROJECT_DIR = "/content/kaizen_os"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

try:
    GRPO_OUTPUT
    SFT_OUTPUT
except NameError:
    SFT_OUTPUT  = os.path.join(PROJECT_DIR, "kaizen_sft_model")
    GRPO_OUTPUT = os.path.join(PROJECT_DIR, "kaizen_grpo_model")

from agent.llm_agent import LLMAgent
from environment.kaizen_env import KaizenEnv

N_EPISODES = 10

def benchmark(model_path: str, label: str) -> dict:
    print(f"\n{'─'*50}")
    print(f"Benchmarking: {label}")
    print(f"{'─'*50}")

    agent = LLMAgent(model_name=model_path)
    env   = KaizenEnv(broadcast=False)

    results = []
    for ep in range(N_EPISODES):
        obs, _ = env.reset()
        cumulative = 0.0
        steps_taken = 0
        resolved = False

        while True:
            raw, action, err = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(raw)
            cumulative += reward
            steps_taken += 1

            if terminated:
                resolved = obs["active_chaos"] is None
                break
            if truncated:
                break

        results.append({
            "episode": ep + 1,
            "reward": round(cumulative, 3),
            "steps": steps_taken,
            "resolved": resolved,
        })

        status = "✅ RESOLVED" if resolved else "❌ timeout"
        print(f"  Ep {ep+1:02d}: reward={cumulative:+.2f}  steps={steps_taken}  {status}")

    resolved_count = sum(1 for r in results if r["resolved"])
    avg_reward     = sum(r["reward"] for r in results) / len(results)
    avg_steps      = sum(r["steps"] for r in results if r["resolved"]) / max(resolved_count, 1)

    print(f"\n  Resolution rate : {resolved_count}/{N_EPISODES} ({100*resolved_count/N_EPISODES:.0f}%)")
    print(f"  Avg reward      : {avg_reward:+.3f}")
    print(f"  Avg steps (✅)  : {avg_steps:.1f}")

    return {"label": label, "resolution": resolved_count, "avg_reward": avg_reward, "avg_steps": avg_steps}


# Benchmark SFT model
sft_stats = benchmark(SFT_OUTPUT, "SFT model")

# Benchmark GRPO model (if it exists)
if os.path.isfile(os.path.join(GRPO_OUTPUT, "adapter_config.json")):
    grpo_stats = benchmark(GRPO_OUTPUT, "GRPO model")

    print(f"\n{'='*50}")
    print("COMPARISON")
    print(f"{'='*50}")
    print(f"  Resolution : SFT {sft_stats['resolution']}/10  →  GRPO {grpo_stats['resolution']}/10")
    print(f"  Avg reward : SFT {sft_stats['avg_reward']:+.3f}  →  GRPO {grpo_stats['avg_reward']:+.3f}  "
          f"(Δ {grpo_stats['avg_reward'] - sft_stats['avg_reward']:+.3f})")
else:
    print("\nℹ️  GRPO model not found — run GRPO training cell first to compare.")


## 🔁  Continual Improvement Guide

To keep improving the model across sessions:

### Within the same Colab session
Just re-run cells **6️⃣** (SFT) and **7️⃣** (GRPO) — the scripts automatically detect and load the existing adapters.

### Across Colab sessions (with Google Drive)
1. Enable `USE_DRIVE = True` in cell **5️⃣** and mount Drive
2. Each session re-mounts Drive and loads the adapters from there
3. Loss/reward plots accumulate across all runs in `sft_combined_runs.png` and `grpo_combined_runs.png`

### Crash recovery
If Colab crashes mid-training:
- Checkpoints are saved every 25 steps (SFT) or 20 steps (GRPO)
- Re-running the training cell automatically resumes from the latest checkpoint
- You lose at most ~25/20 steps of work

### Run history
```
kaizen_sft_model/run_history.json   — all SFT runs with loss values
kaizen_grpo_model/run_history.json  — all GRPO runs with reward values
```

### Tips for faster improvement
- Increase `MAX_STEPS` in the training cells for longer runs (200 SFT / 150 GRPO)
- Add more golden examples to `training/golden_examples.json` (aim for 100+)
- After several GRPO runs, reduce `KL_COEF` from 0.1 to 0.05 to allow more divergence from the SFT prior


## 💾  Download Plots & Model

In [ ]:
import os, glob, zipfile
from google.colab import files

try:
    SFT_OUTPUT
    GRPO_OUTPUT
except NameError:
    SFT_OUTPUT  = "/content/kaizen_os/kaizen_sft_model"
    GRPO_OUTPUT = "/content/kaizen_os/kaizen_grpo_model"

DOWNLOAD = "plots"   # "plots", "models", or "all"

zip_path = "/content/kaizen_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:

    # Always include plots
    for pattern in [
        os.path.join(SFT_OUTPUT,  "*.png"),
        os.path.join(GRPO_OUTPUT, "*.png"),
        os.path.join(SFT_OUTPUT,  "run_history.json"),
        os.path.join(GRPO_OUTPUT, "run_history.json"),
    ]:
        for p in glob.glob(pattern):
            zf.write(p, arcname=os.path.relpath(p, "/content/kaizen_os"))
            print(f"  + {os.path.relpath(p, '/content/kaizen_os')}")

    # Include adapter weights if requested
    if DOWNLOAD in ("models", "all"):
        for model_dir in [SFT_OUTPUT, GRPO_OUTPUT]:
            for fname in ["adapter_config.json", "adapter_model.safetensors",
                          "tokenizer.json", "tokenizer_config.json"]:
                fpath = os.path.join(model_dir, fname)
                if os.path.isfile(fpath):
                    zf.write(fpath, arcname=os.path.relpath(fpath, "/content/kaizen_os"))
                    print(f"  + {os.path.relpath(fpath, '/content/kaizen_os')}")

size_mb = os.path.getsize(zip_path) / 1e6
print(f"\n📦  Archive: {zip_path} ({size_mb:.1f} MB)")
files.download(zip_path)
